In [1]:
import numpy as np
import pandas as pd
import os

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

In [ ]:
index_df = pd.read_csv("features/feature_index_w3_s1.csv")

In [ ]:
# Raw features
X_feat = []
y = []

for _, row in index_df.iterrows():
    feats = np.load(row["feature_path"])
    label = 1 if row["label_type"] == "fall" else 0
    X_feat.append(feats)
    y.extend([label] * len(feats))
X_feat = np.vstack(X_feat)
y = np.array(y)

print(X_feat.shape, y.shape)

(143850, 22) (143850,)


In [5]:
# kmeans tokens
X_tok = []

for _, row in index_df.iterrows():
    subject_id = row["subject_id"]
    file_id = row["file_id"]
    token_path = f"tokens/w3_k60_s1/{subject_id}__{file_id}.npy"
    tokens = np.load(token_path)
    X_tok.append(tokens.reshape(-1,1))
X_tok = np.vstack(X_tok)

In [6]:
# SAX dataset
X_sax = []

for _, row in index_df.iterrows():
    subject_id = row["subject_id"]
    file_id = row["file_id"]
    token_path = f"tokens_sax/w3_a5_s1/{subject_id}__{file_id}.npy"
    tokens = np.load(token_path)
    X_sax.append(tokens.reshape(-1,1))
X_sax = np.vstack(X_sax)

In [7]:
Xf_train, Xf_test, y_train, y_test = train_test_split(X_feat, y, test_size=0.2, random_state=42)
Xt_train, Xt_test, _, _ = train_test_split(X_tok, y, test_size=0.2, random_state=42)
Xs_train, Xs_test, _, _ = train_test_split(X_sax, y, test_size=0.2, random_state=42)

In [8]:
def evaluate(X_train, X_test, y_train, y_test, name):

    rf = RandomForestClassifier(n_estimators=100)
    rf.fit(X_train, y_train)
    preds = rf.predict(X_test)

    print(name)
    print("Accuracy:", accuracy_score(y_test, preds))
    print("F1:", f1_score(y_test, preds))
    print("-----")

In [9]:
evaluate(Xf_train, Xf_test, y_train, y_test, "RAW FEATURES")
evaluate(Xt_train, Xt_test, y_train, y_test, "LEARNED TOKENS")
evaluate(Xs_train, Xs_test, y_train, y_test, "SAX TOKENS")

RAW FEATURES
Accuracy: 0.9803267292318387
F1: 0.9592747157864441
-----
LEARNED TOKENS
Accuracy: 0.8678832116788321
F1: 0.6922017977164143
-----
SAX TOKENS
Accuracy: 0.754257907542579
F1: 0.0
-----


In [10]:
np.mean(y)

0.24984358706986445